# 01 — Functions as Objects

This is the first notebook in the **03_Decorators** section.

Before learning decorator syntax such as `@decorator`, we need to understand a foundational Python idea:

> **Functions are objects.**

Because functions are objects, they can be assigned to variables, passed to other functions, stored in collections, and returned from functions.

These capabilities form the foundation of decorators.

```text
Functions are objects
        ↓
Functions can be passed around
        ↓
Functions can be passed to other functions
        ↓
Functions can return functions
        ↓
Higher-order functions
        ↓
Decorators
```

## 1. Introduction

Start with a normal function:

```python
def greet():
    print("Hello!")
```

The important distinction is:

```text
greet
  ↓
function object

greet()
  ↓
calls the function
```

`greet` refers to the function object. `greet()` calls it.

In [ ]:
def greet():
    print("Hello!")

greet()

In [ ]:
def greet():
    print("Hello!")

print(greet)

## 2. Functions Are Objects

A function created with `def` is an object.

```python
def greet():
    print("Hello!")

print(type(greet))
```

The result is:

```text
<class 'function'>
```

This means `greet` is an object representing the function.

In [ ]:
def greet():
    print("Hello!")

print(type(greet))

## 3. Assigning Functions to Variables

A function can be assigned to another variable.

```python
def greet():
    print("Hello!")

say_hello = greet
```

This does **not** execute `greet`. It creates another reference to the same function object.

In [ ]:
def greet():
    print("Hello!")

say_hello = greet

say_hello()

In [ ]:
def greet():
    print("Hello!")

say_hello = greet

greet()
say_hello()

Think of it as:

```text
greet ───────┐
             ↓
        function object
             ↑
say_hello ───┘
```

Both names refer to the same function object.

## 4. Calling a Function Through Another Variable

A variable referring to a function can be used to call that function.

In [ ]:
def add(a, b):
    return a + b

operation = add

print(operation(10, 20))

In [ ]:
def add(a, b):
    return a + b

operation = add

print(add(10, 20))
print(operation(10, 20))

The two calls use different names, but both names refer to the same function object.

## 5. Checking Function Identity

Use `is` to check whether two names refer to the same object.

In [ ]:
def greet():
    print("Hello!")

another_name = greet

print(greet is another_name)

In [ ]:
def greet():
    print("Hello!")

def another_greet():
    print("Hello!")

print(greet is another_greet)

The first result is `True`; the second is `False`. Identical-looking function code does not mean the functions are the same object.

## 6. Functions as Arguments

Functions can be passed to other functions.

```python
def greet():
    print("Hello!")

def execute(function):
    function()
```

Pass `greet`, not `greet()`:

```text
greet
  ↓
pass the function object

greet()
  ↓
execute the function immediately
```

In [ ]:
def greet():
    print("Hello!")

def execute(function):
    function()

execute(greet)

Notice:

```python
execute(greet)
```

not:

```python
execute(greet())
```

The first passes the function object. The second calls the function first and passes its return value.

## 7. Functions Stored in Collections

Functions can be stored in lists just like other objects.

In [ ]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

operations = [add, subtract]

print(operations[0](10, 5))
print(operations[1](10, 5))

This makes it possible to select behavior dynamically.

## 8. Functions as Dictionary Values

A dictionary can store functions as values.

In [ ]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

operations = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply
}

print(operations["add"](10, 5))
print(operations["multiply"](10, 5))

The dictionary maps names to function objects. This is a practical dispatch-style pattern and will provide useful context for later decorator examples.

## 9. Functions as Return Values

A function can return another function. This is essential groundwork for decorators.

In [ ]:
def create_greeting():
    def greet():
        print("Hello!")

    return greet

message = create_greeting()

message()

The process is:

```text
create_greeting()
       ↓
returns function object
       ↓
message
       ↓
message()
```

`return greet` returns the function object. `return greet()` would call it and return its result.

This notebook only establishes the function-object behavior. The deeper explanation of nested functions and closures was covered earlier in **Functions and Scope**.

## 10. Function Attributes

Functions are objects, so they can have attributes.

In [ ]:
def greet():
    print("Hello!")

greet.description = "Greeting function"

print(greet.description)

In [ ]:
def greet():
    print("Hello!")

print(greet.__name__)

In [ ]:
def greet():
    """Print a greeting message."""
    print("Hello!")

print(greet.__doc__)

Function attributes such as `__name__` and `__doc__` provide useful metadata.

This becomes important later when learning `functools.wraps`, because decorators can otherwise replace a wrapped function's metadata with the wrapper's metadata.

## 11. Understanding Function References

A function reference and a function call are different.

### Reference

```python
x = greet
```

`x` refers to the function object.

### Call

```python
x()
```

This calls the function.

### Passing a function

```python
execute(greet)
```

This passes the function object.

### Calling before passing

```python
execute(greet())
```

This calls `greet` first and passes its return value.

```text
function name
     ↓
reference to function object

function name()
     ↓
call the function
```

In [ ]:
def hello():
    print("Hello!")

def execute(func):
    func()

execute(hello)

Why do we pass `hello` rather than `hello()`?

Because `execute()` needs the function object so that it can call the function itself.

## 12. Common Mistakes

### Mistake 1 — Calling instead of passing

Wrong:

```python
execute(greet())
```

Correct:

```python
execute(greet)
```

The wrong version executes `greet` immediately and passes its return value.

### Mistake 2 — Thinking assignment copies the function

```python
x = greet
```

creates another reference to the same function object; it does not create an independent copy.

### Mistake 3 — Confusing a function name with its returned result

```python
result = greet
```

stores the function reference.

```python
result = greet()
```

calls the function and stores its return value.

### Mistake 4 — Forgetting that functions can be returned

```python
def outer():
    def inner():
        print("Hello!")
    return inner

function = outer()
function()
```

The returned value is a function object that can be called later.

In [ ]:
def get_message():
    return "Hello!"

function_reference = get_message
result = get_message()

print(function_reference)
print(result)

In [ ]:
def outer():
    def inner():
        print("Hello!")
    return inner

function = outer()
function()

## 13. Practical Examples

### Example 1 — Calculator operation

In [ ]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def calculate(operation, a, b):
    return operation(a, b)

print(calculate(add, 10, 5))
print(calculate(subtract, 10, 5))

### Example 2 — Function list

In [ ]:
def square(x):
    return x ** 2

def cube(x):
    return x ** 3

operations = [square, cube]

for operation in operations:
    print(operation(3))

### Example 3 — Returning a function

This example also leads toward closures. For this notebook, focus only on the fact that `create_multiplier()` returns a function object. The deeper explanation of why `number` remains available belongs to the earlier **Nested Functions and Closures** notebook.

In [ ]:
def create_multiplier(number):
    def multiply(value):
        return value * number
    return multiply

double = create_multiplier(2)

print(double(10))

## 14. Summary

### Key Takeaways

- Functions are objects in Python.
- A function can be assigned to another variable.
- Multiple names can refer to the same function.
- Functions can be passed as arguments.
- Functions can be stored in lists and dictionaries.
- Functions can be returned from other functions.
- `func` and `func()` mean different things.
- `func` means the function object.
- `func()` calls the function.
- These capabilities make higher-order functions possible.
- These capabilities also form the foundation of decorators.

### Why This Notebook Comes First

```text
Functions are objects
        ↓
Functions can be passed around
        ↓
Functions can be passed to other functions
        ↓
Functions can return functions
        ↓
Higher-order functions
        ↓
Decorators
```

The next notebooks can build on this foundation rather than jumping directly into `@decorator` syntax.